# 02 · E2 Fusion baselines and E3 fusion weights (ULB)

No training. Loads the component scores saved by notebook 01 and recombines them.

* **E2**: ten configurations: the three detectors alone, the three equal-weight pairs, equal weights over all three, the median of the three, mean percentile rank, and RAREGuard's 0.3/0.3/0.4.
* **E3**: the reference weights against equal weights and IF-, OCSVM- and AE-dominant weights (0.5 / 0.25 / 0.25).

All use the percentile threshold with c = 0.002 on validation scores.

The last section measures how much each detector actually contributes to the fused score (the AE compression issue discussed in the paper).

> **Rebuilt experiment.** The v5 notebook that produced `fusion_baselines_by_seed.csv` was not among the uploaded files. This version follows the method as described in the manuscript. Rank fusion maps each detector's score to its empirical percentile among that detector's **training** scores, then averages. If the original used a different reference for the ranks, Rank_Mean_3 can differ slightly.

| | |
|---|---|
| Trains models | no |
| Needs | notebook 01 (`results/E1_components/scores/`) |
| Writes | `results/E2_E3_fusion/`: `fusion_baselines_by_seed.csv`, `fusion_weights_by_seed.csv`, their `_summary.csv`, `ae_contribution_by_seed.csv` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr

EXP = "E2_E3_fusion"
W = cfg["fusion_weights"]
C = cfg["contamination"]

E2 = {
    "IF_Only":            ("weighted", {"IF": 1.0}),
    "OCSVM_Only":         ("weighted", {"OCSVM": 1.0}),
    "AE_Only":            ("weighted", {"AE": 1.0}),
    "IF_OCSVM_50_50":     ("weighted", {"IF": 0.5, "OCSVM": 0.5}),
    "IF_AE_50_50":        ("weighted", {"IF": 0.5, "AE": 0.5}),
    "OCSVM_AE_50_50":     ("weighted", {"OCSVM": 0.5, "AE": 0.5}),
    "Equal_3":            ("weighted", {"IF": 1/3, "OCSVM": 1/3, "AE": 1/3}),
    "Median_3":           ("median",   None),
    "Rank_Mean_3":        ("rank",     None),
    "RAREGuard_30_30_40": ("weighted", W),
}
E3 = {
    "Original_30_30_40": W,
    "Equal_Weights":     {"IF": 1/3,  "OCSVM": 1/3,  "AE": 1/3},
    "IF_Dominant":       {"IF": 0.50, "OCSVM": 0.25, "AE": 0.25},
    "OCSVM_Dominant":    {"IF": 0.25, "OCSVM": 0.50, "AE": 0.25},
    "AE_Dominant":       {"IF": 0.25, "OCSVM": 0.25, "AE": 0.50},
}
assert all(abs(sum(w.values()) - 1) < 1e-9 for w in E3.values())

def fused(z, rule, weights, part):
    parts = rg.scores_from_npz(z, part)
    if rule == "weighted":
        return rg.weighted_fusion(parts, weights)
    if rule == "median":
        return rg.median_fusion(parts)
    if rule == "rank":
        return rg.rank_fusion(parts, rg.scores_from_npz(z, "train"))
    raise ValueError(rule)

## Evaluate every configuration on the same scores

In [ ]:
run = rg.Run(cfg, EXP, "02_E2_E3_fusion.ipynb", "Fusion baselines (E2) and fusion weights (E3)")
e2_rows, e3_rows, contrib_rows = [], [], []

for seed in cfg["seeds"]:
    z = rg.load_scores(cfg, "E1_components", f"ulb_seed{seed}")
    for name, (rule, weights) in E2.items():
        val_s, test_s = fused(z, rule, weights, "val"), fused(z, rule, weights, "test")
        thr = rg.percentile_threshold(val_s, C)
        e2_rows.append({"Seed": seed, "Configuration": name, "Rule": rule,
                        **rg.evaluate(z["y_test"], test_s, thr, val_s)})
    for name, weights in E3.items():
        val_s, test_s = fused(z, "weighted", weights, "val"), fused(z, "weighted", weights, "test")
        thr = rg.percentile_threshold(val_s, C)
        e3_rows.append({"Seed": seed, "Configuration": name,
                        "IF_Weight": weights["IF"], "OCSVM_Weight": weights["OCSVM"], "AE_Weight": weights["AE"],
                        **rg.evaluate(z["y_test"], test_s, thr, val_s)})

    # How much does each detector move the fused score? (test partition)
    parts = rg.scores_from_npz(z, "test")
    fused_test = rg.weighted_fusion(parts, W)
    spread = {m: W[m] * parts[m].std() for m in parts}
    row = {"Seed": seed}
    for m in parts:
        row[f"{m}_Median"] = float(np.median(parts[m]))
        row[f"{m}_SD"] = float(parts[m].std())
        row[f"{m}_Share_of_Weighted_Spread"] = spread[m] / sum(spread.values())
        row[f"Spearman_Fused_vs_{m}"] = spearmanr(fused_test, parts[m]).correlation
    contrib_rows.append(row)
    print(f"seed {seed} done")

e2, e3, contrib = pd.DataFrame(e2_rows), pd.DataFrame(e3_rows), pd.DataFrame(contrib_rows)
run.save_csv(e2, "fusion_baselines_by_seed.csv")
run.save_csv(rg.mean_sd(e2, ["Configuration"], rg.METRICS + ["TP", "FP", "Test_Alert_Rate"]), "fusion_baselines_summary.csv")
run.save_csv(e3, "fusion_weights_by_seed.csv")
run.save_csv(rg.mean_sd(e3, ["Configuration"], rg.METRICS + ["TP", "FP", "Test_Alert_Rate"]), "fusion_weights_summary.csv")
run.save_csv(contrib, "ae_contribution_by_seed.csv")
run.finish()

## Checks

In [ ]:
# The reference fusion here must reproduce RAREGuard_Full from E1 exactly (same scores, same threshold).
e1 = rg.read_result(cfg, "E1_components", "runs_by_seed.csv")
e1 = e1[(e1.Experiment == "RAREGuard_Full") & (e1.Threshold_Mode == "Fully unsupervised")].set_index("Seed")
ref = e2[e2.Configuration == "RAREGuard_30_30_40"].set_index("Seed")
assert np.allclose(ref.loc[e1.index, "F1"], e1.F1) and np.allclose(ref.loc[e1.index, "PR_AUC"], e1.PR_AUC, atol=1e-6)
print("E2 reference matches E1 RAREGuard_Full on every seed.")

summary = rg.read_result(cfg, EXP, "fusion_baselines_summary.csv")
display(summary.sort_values("PR_AUC_mean", ascending=False)[["Configuration", "F1_mean", "F1_sd", "PR_AUC_mean", "PR_AUC_sd"]].round(4))
display(contrib.filter(like="Share").describe().loc[["mean", "min", "max"]].round(3))